# 再現できるバックテストを運用する

バックテストは過去についての主張であり、その価値は裏づけとなる証拠の価値と同じです。半年後、
レビューする人はもう一度走らせて同じ数字を得たいと思います。

やっかいなのは、調査用データベースが取り込みを続けることです。遅れてプリントが届き、ベンダーが
数値を訂正し、同じ実行が前より大きなテーブルを読むようになります。コードをピン留めするだけでは
足りません。足元でデータが動いたからです。

再現性は乱数シードの話ではなく、運用の性質です。このレシピでは、マーケットデータの断面をピン留め
し、遅れて届いたデータを追記し、ピン留めした実行が変わらないことを示し、レビューに要る実行
マニフェストを確認します。

## ここで使う用語

| 用語 | 意味 |
| --- | --- |
| 再現性 | 数か月後に再実行しても同じバイト列を読み、同じ数字が出ること |
| ピン留め | 実行の入力を特定のバージョンやスナップショットに固定すること |
| 遅れて届くデータ | ある期間の行が、その期間をすでに読んだあとで到着すること |
| カバレッジ・ゲート | ピン留めした窓に、実行が前提としたデータが実際に入っているかの検査 |
| 実行マニフェスト | その実行が何を読み何を出したかの記録。レビューが読むのはこれ |

はじめて見る用語があれば、[GLOSSARY.ja.md](../../GLOSSARY.ja.md) にもう少し詳しい説明があります。ほかのレシピで使う用語もまとめてあります。

240秒ぶんのテープを1本作り、承認済みの180秒の断面と、遅れて届く末尾に分けます。実行が認定された
あとも取り込みを続ける調査用データベースを模したものです。

| 入力 | 行数 | 役割 |
|---|---:|---|
| `instruments` | 2 | 取引所と契約の安定したメタデータ |
| 初期の `book_deltas` | 360 | 承認済みの L2 断面 |
| 初期の `trades` | 180 | 承認済みのプリント断面 |
| 遅れて届く末尾 | 板360行 + 約定60件 | あとから続く取り込み |

In [1]:
import datetime as dt

import pandas as pd
import pyarrow.compute as pc

import h5i_db
from h5i_db import backtest
import cookbook_utils as cu

full = cu.make_backtest_fixture(steps=240)
base = dt.datetime(2026, 6, 1, 14, 0, 0)
cutoff = base + dt.timedelta(seconds=181)

initial_book = full["book_deltas"].filter(
    pc.less(full["book_deltas"]["ts_init"], cutoff)
)
late_book = full["book_deltas"].filter(
    pc.greater_equal(full["book_deltas"]["ts_init"], cutoff)
)
initial_trades = full["trades"].filter(pc.less(full["trades"]["ts_init"], cutoff))
late_trades = full["trades"].filter(
    pc.greater_equal(full["trades"]["ts_init"], cutoff)
)

print(f"initial book: {initial_book.num_rows:,} rows")
print(f"late book: {late_book.num_rows:,} rows")
initial_book.to_pandas().tail()

initial book: 360 rows
late book: 120 rows


,ts_init,ts_event,instrument_id,outcome,action,side,price,size,event_index,is_last,source_vendor
355,2026-06-01 14:02:58,2026-06-01 14:02:58,RATE-CUT-YES,0,snapshot,sell,0.5500,110.0,178,True,cookbook-sim
356,2026-06-01 14:02:59,2026-06-01 14:02:59,RATE-CUT-YES,0,snapshot,buy,0.5416,120.0,179,False,cookbook-sim
357,2026-06-01 14:02:59,2026-06-01 14:02:59,RATE-CUT-YES,0,snapshot,sell,0.5516,120.0,179,True,cookbook-sim
358,2026-06-01 14:03:00,2026-06-01 14:03:00,RATE-CUT-YES,0,snapshot,buy,0.5432,80.0,180,False,cookbook-sim
359,2026-06-01 14:03:00,2026-06-01 14:03:00,RATE-CUT-YES,0,snapshot,sell,0.5532,80.0,180,True,cookbook-sim


承認済みの断面を作って読み込みます。初期バッチと後続バッチでテーブルのスキーマが同じなので、
append が1本の論理的な履歴を保てます。

In [2]:
db = h5i_db.Database(
    cu.fresh_db("04_reproducible_backtest_operations"),
    create=True,
)
db.create_table(
    "instruments",
    full["instruments"].schema,
    time_column="ts_init",
)
db.create_table("book_deltas", initial_book.schema, time_column="ts_init")
db.create_table("trades", initial_trades.schema, time_column="ts_init")
db.append("instruments", full["instruments"], note="reference data")
db.append("book_deltas", initial_book, note="approved 180-second book cut")
db.append("trades", initial_trades, note="approved 180-second trade cut")
db.snapshot(
    "approved-cut",
    tables=["instruments", "book_deltas", "trades"],
    note="Input approved by research controls",
)
db.versions("book_deltas")

[{'sequence': 0,
  'op': 'create',
  'committed_at_ns': 1785449400869704049,
  'rows': 0,
  'bytes': 0,
  'segments': 0,
  'execution_mode': 'direct'},
 {'sequence': 1,
  'op': 'append',
  'committed_at_ns': 1785449400906436129,
  'rows': 360,
  'bytes': 8283,
  'segments': 1,
  'note': 'approved 180-second book cut',
  'execution_mode': 'direct'}]

戦略は承認済みの断面の中で建玉を作り、遅れて届く末尾で決済します。どの実行でも同じ signals
テーブルを使うので、変わるのはマーケットデータの読み出し地点だけです。

| 列 | 型 | 意味 |
|---|---|---|
| `ts` | `timestamp[ns]` | 注文意図が到着する時刻 |
| `instrument_id` | `string` | 契約の識別子 |
| `side` | `string` | 買いで建て、売りで決済 |
| `quantity` | `float64` | 要求する数量 |
| `tag` | `string` | 安定した監査用ラベル |

In [3]:
signals = backtest.signal_table(
    [
        {
            "ts": base + dt.timedelta(seconds=30),
            "instrument_id": "RATE-CUT-YES",
            "side": "buy",
            "quantity": 50.0,
            "tag": "open-approved",
        },
        {
            "ts": base + dt.timedelta(seconds=210),
            "instrument_id": "RATE-CUT-YES",
            "side": "sell",
            "quantity": 50.0,
            "tag": "close-late",
        },
    ]
)
print(f"{signals.num_rows:,} rows x {signals.num_columns} columns")
signals.to_pandas()

2 rows x 10 columns


,ts,instrument_id,outcome,side,quantity,kind,limit_price,time_in_force,tag,reduce_only
0,2026-06-01 14:00:30,RATE-CUT-YES,0,buy,50.0,market,NaN,NaN,open-approved,False
1,2026-06-01 14:03:30,RATE-CUT-YES,0,sell,50.0,market,NaN,NaN,close-late,False


戦略の意図は、マーケットのスナップショットを作ったあとに保存します。実行フォークは戦略テーブルを
別にピン留めするので、スナップショットはマーケットデータの承認地点として純粋なまま残ります。

In [4]:
backtest.create_signal_table(db)
db.append("signals", signals, note="approved strategy intent")

{'table': 'signals',
 'sequence': 1,
 'op': 'append',
 'rows_total': 2,
 'segments_total': 1,
 'segments_added': 1,
 'segments_deduped': 0,
 'committed_at_ns': 1785449400976105933}

遅れたデータを取り込む前に、承認済みの断面に対して実行します。ピン留めしたデータでリプレイが
終わるため、到達するのは建玉のシグナルだけです。

In [5]:
first = backtest.run(
    db,
    "approved-before-late-data",
    starting_cash=10_000.0,
    signals="signals",
    snapshot="approved-cut",
    equity_interval_nanos=10_000_000_000,
)
first

{'run_id': 'approved-before-late-data',
 'fork': 'bt-approved-before-late-data',
 'digest': 'e9d3cff467107b62fbdb005189f3a5e5d289eba406d16f3e63f55245cd9e4543',
 'starting_cash': 10000.0,
 'final_cash': 9974.47,
 'realized_pnl': 0.0,
 'commissions': 0.0,
 'funding_paid': 0.0,
 'fills': 1,
 'orders': 1,
 'records_processed': 360,
 'simulated_through_ns': 1780322580000000000,
 'equity_points': 19,
 'settlement_applied': False,
 'coverage': None,
 'liquidations': 0,
 'rejected_for_margin': 0,
 'self_trades_prevented': 0,
 'calibration_samples': [],
 'set_operations': [],
 'forecasts': 0,
 'mark_points': 19,
 'expirations': [],
 'metrics': {'orders_submitted': 1,
  'orders_filled': 1,
  'orders_cancelled_unfilled': 0,
  'orders_rejected_margin': 0,
  'orders_rejected_risk': 0,
  'orders_rejected_self_trade': 0,
  'orders_rejected_naked_short': 0,
  'orders_rejected_expired': 0,
  'fills_taker': 1,
  'fills_maker': 0,
  'book_gaps': 0,
  'liquidations': 0,
  'set_operations': 0,
  'set_opera

遅れて届いた末尾を、ふつうの取り込みとして追記します。既存のバージョンと名前付きスナップショットは
そのまま読めますし、スナップショットにデータが複製されることもありません。

In [6]:
db.append("book_deltas", late_book, note="late-arriving final minute")
db.append("trades", late_trades, note="late-arriving final minute")
print(db.versions("book_deltas")[-1])

{'sequence': 2, 'op': 'append', 'committed_at_ns': 1785449401168986889, 'rows': 480, 'bytes': 13672, 'segments': 2, 'note': 'late-arriving final minute', 'execution_mode': 'direct'}


承認済みスナップショットで再実行し、あわせて最新のデータでも1回実行します。ピン留めした結果は
変わらないはずです。最新での実行は決済シグナルまで到達できるので、調査の入力としては別物です。

In [7]:
pinned_again = backtest.run(
    db,
    "approved-after-late-data",
    starting_cash=10_000.0,
    signals="signals",
    snapshot="approved-cut",
    equity_interval_nanos=10_000_000_000,
)
latest = backtest.run(
    db,
    "latest-after-late-data",
    starting_cash=10_000.0,
    signals="signals",
    equity_interval_nanos=10_000_000_000,
)

comparison = pd.DataFrame(
    [
        {"run": "pinned before append", **first},
        {"run": "pinned after append", **pinned_again},
        {"run": "latest after append", **latest},
    ]
).set_index("run")
comparison[
    [
        "fills",
        "orders",
        "records_processed",
        "final_cash",
        "realized_pnl",
    ]
]

,fills,orders,records_processed,final_cash,realized_pnl
run,,,,,
pinned before append,1,1,360,9974.47,0.00
pinned after append,1,1,360,9974.47,0.00
latest after append,2,2,480,10002.48,2.48


レビューする人が気にする証拠を、そのまま表明します。ピン留めした実行どうしは、金額もイベント数も
一致します。最新での実行は、意図して違う結果になります。

In [8]:
stable_fields = (
    "fills",
    "orders",
    "records_processed",
    "final_cash",
    "realized_pnl",
    "commissions",
)
assert all(first[field] == pinned_again[field] for field in stable_fields)
assert latest["records_processed"] > first["records_processed"]
assert latest["fills"] > first["fills"]
print("Pinned result survived subsequent ingestion unchanged.")

Pinned result survived subsequent ingestion unchanged.


カバレッジ・ゲートは、不完全なデータをその場での失敗に変えます。ここでは、承認済みの断面が、
遅れて届く末尾まで伸びる窓を満たせません。

In [9]:
try:
    backtest.run(
        db,
        "coverage-must-fail",
        starting_cash=10_000.0,
        signals="signals",
        snapshot="approved-cut",
        window=(
            base + dt.timedelta(seconds=1),
            base + dt.timedelta(seconds=240),
        ),
        minimum_coverage=0.95,
    )
except h5i_db.InvalidInputError as error:
    print(f"Rejected as intended: {error}")
else:
    raise AssertionError("the incomplete approved cut passed its coverage gate")

Rejected as intended: [invalid_input] coverage 74.9% of [1780322401000000000, 1780322640000000000) is below the required 95.0%; the window loaded [1780322401000000000, 1780322580000000001) and is missing 59999999999 ns


実行フォークにはそれぞれ、1行のマニフェストと詳細な結果テーブルが入っています。マニフェスト、
ソースのスナップショット名、戦略のバージョン、設定の4つは、レビュー用の資料に残してください。
約定テーブルが執行についての正本であることは変わりません。

In [10]:
audit_rows = []
for label, report in (
    ("pinned-before", first),
    ("pinned-after", pinned_again),
    ("latest", latest),
):
    run_db = db.fork(report["fork"])
    manifest = run_db.read("bt_run").to_pandas().iloc[0].to_dict()
    manifest["label"] = label
    manifest["fork"] = report["fork"]
    audit_rows.append(manifest)
    run_db.close()
audit = pd.DataFrame(audit_rows).set_index("label")
audit[
    [
        "run_id",
        "config_digest",
        "records_processed",
        "final_cash",
        "realized_pnl",
        "fork",
    ]
]

,run_id,config_digest,records_processed,final_cash,realized_pnl,fork
label,,,,,,
pinned-before,approved-before-late-data,e9d3cff467107b62fbdb005189f3a5e5d289eba406d16f...,360,9974.47,0.00,bt-approved-before-late-data
pinned-after,approved-after-late-data,12f787c49cafa9216a356ff23148b1ce5ffc74e9412420...,360,9974.47,0.00,bt-approved-after-late-data
latest,latest-after-late-data,6162db326c6a1fca96c66b6df1eabb3953f6d2df70f579...,480,10002.48,2.48,bt-latest-after-late-data


## まとめ

- バックテストの結果を受け入れる前に、マーケットデータをピン留めする。
- 戦略の意図は、過去データの断面とは別にバージョン管理する。
- 名前付きスナップショットでの再実行は、あとから追記があっても安定している。
- カバレッジ・ゲートは、それらしい指標が出てしまう前に、切り詰められた窓を弾く。
- 実行フォーク、マニフェスト、ダイジェスト、約定は、1つのレビュー可能な成果物として保存する。

In [11]:
db.close()